# Microscope calibration from beam images

**TemGym truth model → ray images → measurements from pixels → independent fit → geometry check.**

The proposed acquisition uses **356 images at 200/195 kV**: the **50 µm aperture for projector calibration**, and **20/50 µm apertures for condenser measurements and prediction checks**. Objective current stays fixed. Detector scale and specimen-plane shifts/tilts are assumed calibrated.

The normal settings give nearly parallel specimen illumination and focused **40,000×/45,000×** detector images. Projector calibration holds one illumination setting per voltage, applies signed **X shifts, X tilts and Y tilts**, and limits each projector lens to **±30%** of its baseline control. Perturbed views need not remain focused or parallel. The condenser design-curve settings are unchanged.

For comparison, this notebook simulates both apertures throughout (**572 images**). The selected fit ignores the 20 µm projector images; those extra images would not be needed for the proposed acquisition. Native TemGym components generate the ray images, including aperture transmission and detector blur. Diffraction is parked for this test.

In [1]:
%matplotlib widget

import os
os.environ["XLA_PYTHON_CLIENT_PREALLOCATE"] = "false"

from pathlib import Path
import jax
jax.config.update("jax_enable_x64", True)
from temgym_core import calibration

print("JAX devices:", jax.devices())
reference_directory = Path("calibration_results/tilt_y_30_ray_acquisition")
output_directory = Path("../generated/microscope_calibration_tilt_y_notebook")
# Leave as None for simulated images; otherwise use an imported dataset folder.
recorded_directory = None


JAX devices: [CudaDevice(id=0)]


**1. Make the images.** Supply the saved acquisition, an output folder, a ray count and a noise seed. The acquisition contains the synthetic truth and recorded controls; the fitter will receive only the measured data and calibrations.

Each image has $10^8$ incident electrons, Poisson counting noise and detector read noise. Increasing the ray count improves numerical accuracy without changing exposure. The 256×256 detector uses 32 internal samples per pixel axis; the detector blur retains its faint tails. This finer grid resolves the small spots produced by some lens perturbations.

For calibrated image arrays, use `calibration.import_images` once, then set `recorded_directory` above. The remaining measurement, browser and fitting cells are the same. See the [array input example](../../src/temgym_core/calibration/README.md#recorded-image-arrays). Camera conversion and a microscope pilot are deferred.

In [2]:
if recorded_directory is None:
    images = calibration.simulate_ray_experiment(
        reference_directory=reference_directory,
        output_directory=output_directory,
        number_of_rays=4194304,
        detector_samples_per_pixel=32,
        add_noise=True,
        random_seed=262510,
    )
else:
    images = Path(recorded_directory)


Running generation; progress: ../generated/microscope_calibration_tilt_y_notebook/generation.log
Completed generation


**2. Measure the images.** Supply the saved pixels. TemGym subtracts the known background and measures the beam centre, X/Y variances, XY covariance and total intensity. It estimates their uncertainties from the detector noise model.

A 16-pixel margin keeps the faint tails of broad beams. These measurements come from the images; extraction does not read the true lens or source parameters.

In [3]:
measurements = calibration.measure_images(experiment_directory=images)

Running measurement; progress: ../generated/microscope_calibration_tilt_y_notebook/measurement.log
Completed measurement


**Inspect the dataset.** The browser displays the pixels used for fitting beside the native ray diagram. Select **Source → condenser mini** to see the first crossover between CL1 and CL3. The diagram can show truth, the initial guess or a saved fit.

Use the sliders to compare condenser settings and aperture sizes. Projector calibration uses equal and opposite X shifts, X tilts and Y tilts; tilt amplitude is reduced where needed to keep the beam on the camera.

After fitting, rerun this browser cell to add the recovered model to the diagram.

Select **50 µm** to view the projector images used by the chosen fit. The 20 µm projector images remain available for comparison.

In [4]:
from importlib import reload
from temgym_core.calibration import viewer

browser = reload(viewer).browse_images(experiment_directory=measurements)
browser

**3. Fit the microscope.** Native `column_matrix` obtains the transfer matrix by differentiating TemGym ray propagation through the components. For a ray $[x,y,\theta_x,\theta_y,1]^T$, TemGym supplies

$$
\begin{pmatrix}
A_{xx}&A_{xy}&B_{xx}&B_{xy}&a_x\\
A_{yx}&A_{yy}&B_{yx}&B_{yy}&a_y\\
C_{xx}&C_{xy}&D_{xx}&D_{xy}&b_x\\
C_{yx}&C_{yy}&D_{yx}&D_{yy}&b_y\\
0&0&0&0&1
\end{pmatrix}.
$$

Known specimen shifts and tilts give $\Delta\bar{\mathbf{x}}_D=A\Delta\mathbf{x}_S+B\Delta\boldsymbol{\theta}_S$. These initialize the projector fit. Beam shape and aperture-pair intensity then initialize the condenser fit. Finally, refine both halves against the original measurement values.

The model is round and coaxial. X shifts and X/Y tilts constrain its scale and rotation response; that symmetry is an assumption of this experiment. The lower blocks $C,D$ follow from the fitted physical lens model. For shape, $H=[A\ B]$ gives $\Sigma_D=H\Sigma_S H^T+\Sigma_{\mathrm{det}}$, where $\Sigma_{\mathrm{det}}$ accounts for calibrated detector blur and pixel sampling.

Fit all 47 physical/source/aperture coordinates with the existing JAX least-squares solver. The first source crossover must remain between CL1's exit and CL3's entrance. This is an operating assumption, not a known distance. Objective current remains fixed; geometry and lens strengths remain unknown.

The fitter predicts these statistics with native matrices and aperture integrals, so it does not need to render images at every iteration. It starts from independent guesses and never reads the generating parameters. Aperture diffraction is disabled for this geometric image test.

The final fit uses ordinary residual/step/gradient stopping criteria. Pixel-derived measurements are not exact equations, even when detector noise is off.

One projector aperture supplies the steering response and beam shape. Its unknown incident dose is fitted out, so absolute intensity is not treated as a calibration. Condenser measurements retain both apertures and their intensity ratio. Set `projector_aperture_um=None` below to compare a fit using both projector apertures.

In [5]:
microscope = calibration.fit_microscope(
    measurements_directory=measurements,
    number_of_starts=4,
    projector_aperture_um=50,
)

Running fitting; progress: ../generated/microscope_calibration_tilt_y_notebook/fitting.log
Completed fitting


**4. Check the fit.** Predict measurements at six unused condenser settings, then compare the recovered geometry with truth. The check below compares image measurements; it does not render another image stack.

Read the geometry errors and uncertainty column as well as the residual. A good fit to the images alone does not establish unique physical geometry.

Imported data need no truth file: the check reports predicted measurements and local geometry uncertainty, leaving truth and error blank.

In [6]:
checks = calibration.check_microscope(
    experiment_directory=measurements,
    microscope=microscope,
    render_images=False,
)
calibration.show_result(checks=checks)

Running checks; progress: ../generated/microscope_calibration_tilt_y_notebook/checks.log
Completed checks
Fit RMS: 0.881; prediction RMS: 1.088


| Parameter | Fitted | Truth | Error | Standard deviation |
|---|---:|---:|---:|---:|
| Objective position (mm) | 2.30018 | 2.3 | 0.000178451 | 0.000213918 |
| IL1 position (mm) | 213.599 | 213.524 | 0.0750031 | 0.0990463 |
| IL2 position (mm) | 290.612 | 290.48 | 0.132343 | 0.182205 |
| IL3 position (mm) | 353.171 | 352.98 | 0.190899 | 0.268582 |
| PL1 position (mm) | 408.234 | 407.983 | 0.250992 | 0.358124 |
| Objective length (mm) | 0.400022 | 0.4 | 2.20462e-05 | 3.8785e-05 |
| IL1 length (mm) | 5.00294 | 5 | 0.00294274 | 0.00459294 |
| IL2 length (mm) | 5.00417 | 5 | 0.00417485 | 0.00625526 |
| IL3 length (mm) | 5.00513 | 5 | 0.00513278 | 0.00759789 |
| PL1 length (mm) | 1.00114 | 1 | 0.00114018 | 0.00153041 |
| Detector position (mm) | 719.163 | 718.75 | 0.413175 | 0.562949 |
| source_z_mm | -324.989 | -325 | 0.0105165 | 0.336902 |
| CL1_z_mm | -124.976 | -125 | 0.0242735 | 0.0531258 |
| CL3_z_mm | -84.9805 | -85 | 0.0195255 | 0.046063 |
| mini_z_mm | -24.9923 | -25 | 0.00772387 | 0.0187523 |
| prefield_z_mm | -2.49959 | -2.5 | 0.000405752 | 0.00124311 |
| CL1_length_mm | 4.99935 | 5 | -0.00064963 | 0.00397968 |
| CL3_length_mm | 4.99925 | 5 | -0.000745835 | 0.00203475 |
| mini_length_mm | 2.9995 | 3 | -0.000504392 | 0.00221097 |
| prefield_length_mm | 0.997692 | 1 | -0.00230833 | 0.00438957 |
| aperture_z_mm | -79.9813 | -80 | 0.0187252 | 0.0447526 |
| aperture_x_um | 0.200204 | 0.2 | 0.000203842 | 0.000287256 |
| aperture_y_um | -0.149866 | -0.15 | 0.000133808 | 0.000352094 |
| Source z, second voltage (mm) | -325.039 | -325.05 | 0.0105774 | 0.337028 |
| Second aperture x (µm) | 0.119833 | 0.12 | -0.000167235 | 0.000369362 |
| Second aperture y (µm) | -0.080368 | -0.08 | -0.000368043 | 0.000394251 |

**What does this show?** This 356-image synthetic fit gives source-z error **0.01049 mm**, with local standard deviation **0.337 mm**. Prediction RMS is **1.088 noise units**. Several projector positions still miss the 0.1 mm target.

**Repeatability:** All six preselected noise realizations produced accepted fits at the same exposure. Source-z scatter was **0.307 mm**, compared with **0.337 mm** predicted uncertainty; the largest error was **0.663 mm**. The [repeatability report](../generated/calibration_repeat_validation/README.md) retains every starting guess, including failures. Each repeat was fitted without a truth file. Six repeats support the synthetic uncertainty estimate; they do not establish global uniqueness or real-microscope accuracy.

The [aperture and perturbation comparison](../generated/microscope_calibration_tilt_y/README.md) explains the 30% limit and single projector aperture. Camera conversion and the microscope pilot remain deferred.

## Why recovering the microscope is difficult

Each image constrains the combined action of many lenses and free-space gaps. Recovering the microscope means finding the individual geometry, field strengths, and source distributions that reproduce this action across the recorded currents and voltages.

Write a ray as $\mathbf{q}=(x,y,\theta_x,\theta_y)^\mathsf{T}$. A free-space gap $z$ and a native hard-edge solenoid of length $\ell$ and signed strength $q$ have matrices

$$
F(z)=\begin{pmatrix}I_2&zI_2\\0&I_2\end{pmatrix},
\qquad
L(\ell,q)=\begin{pmatrix}
\cos q\,R(-q)&\ell\,\dfrac{\sin q}{q}R(-q)\\
-\dfrac{q\sin q}{\ell}R(-q)&\cos q\,R(-q)
\end{pmatrix},
$$

$$
R(-q)=\begin{pmatrix}\cos q&\sin q\\-\sin q&\cos q\end{pmatrix}.
$$

The limit $\sin q/q\to1$ applies at $q=0$. These are the matrices used by `HardEdgeSolenoid`; TemGym embeds the affine ray map in a $5\times5$ matrix when translations or steering are present. At acquisition setting $j$, each lens strength obeys

$$
q_{i,j}=q_{i,0}\,u_{i,j}\,\frac{p(V_0)}{p(V_j)},
$$

where the relative current $u_{i,j}$ and voltage $V_j$ are recorded, while the reference strength $q_{i,0}$ and field length $\ell_i$ are fitted. Here $p(V)$ is the relativistic electron momentum.

Read the following products from right to left. The distances $z_{C,i}$ and $z_{P,i}$ are the free-space gaps in the condenser and projector sections, respectively. The source-to-specimen map contains four solenoid sections,

$$
C_j=
F(z_{C,4})L_{\mathrm{Obj,pre},j}
F(z_{C,3})L_{\mathrm{Mini},j}
F(z_{C,2})L_{\mathrm{CL3},j}
F(z_{C,1})L_{\mathrm{CL1},j}F(z_{C,0,V_j}),
$$

and the specimen-to-detector map contains five,

$$
P_j=
F(z_{P,5})L_{\mathrm{PL1},j}
F(z_{P,4})L_{\mathrm{IL3},j}
F(z_{P,3})L_{\mathrm{IL2},j}
F(z_{P,2})L_{\mathrm{IL1},j}
F(z_{P,1})L_{\mathrm{Obj,post},j}F(z_{P,0}).
$$

The objective pre-field and post-field are separate fitted sections of the same objective region. At zero steering, the geometric map is $M_j=P_jC_j$: **nine solenoid matrices and eleven drift matrices, written as twenty factors**. The two drifts adjoining the specimen can be combined when no kick is applied there. With a calibrated specimen kick $\mathbf{k}_j$, the ray coordinates obey

$$
\mathbf{q}_{D,j}=P_j\bigl(C_j\mathbf{q}_{s,V_j}+\mathbf{k}_j\bigr).
$$

The aperture lies inside $z_{C,2}$, and its transmission weights the rays passing through that plane. Its effect on beam intensity and moments must therefore be evaluated within the column.

### Even three thin lenses couple the unknowns

A smaller, one-dimensional example makes the parameter coupling visible. A thin lens with focal length $f_i$ and free-space propagation over $z$ have matrices

$$
L(f_i)=\begin{pmatrix}1&0\\-\dfrac{1}{f_i}&1\end{pmatrix},
\qquad
F(z)=\begin{pmatrix}1&z\\0&1\end{pmatrix}.
$$

For three lenses separated by $z_1$ and $z_2$, the full product is

$$
\begin{aligned}
M_{\mathrm{toy}}
&=L(f_3)F(z_2)L(f_2)F(z_1)L(f_1)\\[4pt]
&=\begin{pmatrix}1&0\\-1/f_3&1\end{pmatrix}
\begin{pmatrix}1&z_2\\0&1\end{pmatrix}
\begin{pmatrix}1&0\\-1/f_2&1\end{pmatrix}
\begin{pmatrix}1&z_1\\0&1\end{pmatrix}
\begin{pmatrix}1&0\\-1/f_1&1\end{pmatrix}\\[4pt]
&=\begin{pmatrix}A&B\\C&D\end{pmatrix}.
\end{aligned}
$$

Multiplying out all five matrices gives

$$
\begin{aligned}
A={}&1-\frac{z_1}{f_1}-\frac{z_2}{f_1}-\frac{z_2}{f_2}
+\frac{z_1z_2}{f_1f_2},\\[6pt]
B={}&z_1+z_2-\frac{z_1z_2}{f_2},\\[6pt]
C={}&-\frac{1}{f_1}-\frac{1}{f_2}-\frac{1}{f_3}
+\frac{z_1}{f_1f_2}+\frac{z_1}{f_1f_3}\\
&+\frac{z_2}{f_1f_3}+\frac{z_2}{f_2f_3}
-\frac{z_1z_2}{f_1f_2f_3},\\[6pt]
D={}&1-\frac{z_1}{f_2}-\frac{z_1}{f_3}-\frac{z_2}{f_3}
+\frac{z_1z_2}{f_2f_3}.
\end{aligned}
$$

These four expressions were checked with SymPy. The lower-left entry contains products of all three inverse focal lengths and both spacings, including $-z_1z_2/(f_1f_2f_3)$, so a change in one lens can be partly compensated by changes elsewhere. The actual microscope uses the finite-length, rotating solenoid matrices above, whose entries also contain trigonometric functions of the fitted strengths.

The derivative of the full product retains the same coupling. For $M=E_n\cdots E_1$, where each $E_i$ is a lens or free-space matrix,

$$
\frac{\partial M}{\partial\theta_a}
=
\sum_{i=1}^{n}
E_n\cdots E_{i+1}
\frac{\partial E_i}{\partial\theta_a}
E_{i-1}\cdots E_1.
$$

A parameter's effect at the detector depends on the transport both before and after the affected element; the optimizer differentiates this dependence with JAX.

### The 47 fitted parameters

The current two-voltage, two-aperture fit has the following independent coordinates. Positive lengths and strength magnitudes are represented logarithmically in parts of the implementation, but this changes their numerical scaling, not their count.

| Part of the microscope | Unknown quantities | Count |
|---|---|---:|
| Projector geometry | Five lens-centre positions and the detector position | 6 |
| Projector fields | Five field lengths and five reference strengths | 10 |
| Condenser geometry | Four shared axial gaps and four field lengths | 8 |
| Condenser fields | Four reference strengths, including the objective pre-field | 4 |
| Source at 200 kV | Source-to-CL1 gap, $x_s,y_s,\theta_{x,s},\theta_{y,s}$, RMS size, and incoherent angular RMS | 7 |
| Source at 195 kV | The same seven quantities, independently fitted | 7 |
| Aperture axial position | One fractional position in the CL3-to-mini gap | 1 |
| Aperture centres | Separate $x,y$ centres for the 20 and 50 µm apertures | 4 |
| **Total** | **16 projector + 31 condenser/source/aperture coordinates** | **47** |

The two source positions are derived from their source-to-CL1 gaps and the shared column geometry; they are already included in the table. The objective current is fixed, while its reference field strengths remain unknown. Aperture diameters, edge width, detector calibration, voltages, current ratios, and specimen kicks are supplied. Deflector positions are assumed and add no fitted coordinates here.

There are also **280 unknown incident doses** in the selected training data: one for each of 216 single-aperture projector settings and one shared by each of 64 condenser aperture pairs. These doses are eliminated analytically at each trial microscope, leaving 47 coordinates for numerical optimization.

### What the images actually constrain

Each image supplies only six measured statistics,

$$
\mathbf{y}_{j,a}=
\bigl(\bar x,\bar y,\operatorname{var}x,
\operatorname{cov}(x,y),\operatorname{var}y,N\bigr)_{j,a}.
$$

At the aperture, the incident Gaussian spatial density $\rho_j(\mathbf{r};\theta)$ and amplitude transmission $t_a(\mathbf{r};\theta)$ give the transmitted fraction

$$
T_{j,a}(\theta)=\int
\rho_j(\mathbf{r};\theta)\,
|t_a(\mathbf{r};\theta)|^2\,d^2\mathbf{r},
\qquad
N_{j,a}=N_{\mathrm{in},j}T_{j,a}(\theta).
$$

The same weighted distribution determines the transmitted mean and covariance, which are transported to the specimen. Let these be $\boldsymbol\mu_{j,a}$ and $\Sigma_{j,a}$, and let $H_j=(I_2\;0)P_j$ select detector position. In detector length units,

$$
\bar{\mathbf{x}}_{D,j,a}=H_j(\boldsymbol\mu_{j,a}+\mathbf{k}_j),
\qquad
\Sigma_{D,j,a}=H_j\Sigma_{j,a}H_j^\mathsf{T}+\Sigma_{\mathrm{det}}.
$$

Here $\Sigma_{\mathrm{det}}$ includes detector blur and the pixel-sampling correction. The active geometric-ray experiment omits aperture diffraction. Lens settings influence intensity by changing the illumination at the aperture; downstream lossless lenses redistribute the transmitted electrons over the detector.

The selected acquisition has **344 training images and 12 withheld check images**, giving $344\times6=2064$ measured training statistics. With one eliminated dose per training setting, the nominal residual degrees of freedom are $2064-280-47=1737$ when the physical Jacobian has rank 47. The uncertainties and correlations between statistics enter through their measured covariance matrices.

The fit minimizes

$$
\chi^2(\theta)=
\min_{N_{\mathrm{in},j}\geq0}
\sum_{j,a}
\left[\mathbf{f}_{j,a}(\theta,N_{\mathrm{in},j})-\mathbf{y}_{j,a}\right]^\mathsf{T}
W_{j,a}^{-1}
\left[\mathbf{f}_{j,a}(\theta,N_{\mathrm{in},j})-\mathbf{y}_{j,a}\right],
$$

subject to the geometry bounds and first-crossover restriction. Here $W_{j,a}$ is the measured $6\times6$ covariance. The projector-response estimates initialize the fit; the final objective uses the original image statistics once.

Although the number of measurements exceeds the number of parameters, different parameter changes can produce nearly identical changes in these statistics. If the columns of the whitened residual Jacobian $J_{ra}=\partial r_r/\partial\theta_a$ are nearly dependent, $J^\mathsf{T}J$ has small eigenvalues and the corresponding parameter combinations have large local uncertainty. Changing currents, voltage, aperture size, and calibrated steering helps distinguish those combinations. This explains how a fit can predict the beam images within their noise while some recovered axial positions remain uncertain.

## How position, beam shape, and intensity constrain the microscope

Each detector image gives **six numbers**,

$$
\boxed{\bar x_d,\quad \bar y_d,\quad
\operatorname{var}(x_d),\quad \operatorname{cov}(x_d,y_d),\quad
\operatorname{var}(y_d),\quad N.}
$$

The first two locate the beam, the next three describe its ellipse, and $N$ is its total background-subtracted electron count. We use $N$ for intensity here and keep $F(z)$ for free-space propagation. The fit adjusts the lens geometry, lens strengths, source, and aperture positions until their predicted measurements agree with these six numbers across the acquisition.

### 1. Position measures the response to a known shift or tilt

For one transverse direction, the specimen-to-detector relation is

$$
\bar x_d=A\bar x_s+B\bar\theta_{x,s}.
$$

Here $A$ maps a specimen position to a detector position, while $B$ maps a specimen angle to a detector position. Comparing two exposures at the same lens setting gives

$$
\boxed{\Delta\bar x_d=A\,\Delta\bar x_s+B\,\Delta\bar\theta_{x,s}.}
$$

A calibrated **pure shift** gives $A=\Delta\bar x_d/\Delta\bar x_s$, and a calibrated **pure tilt** gives $B=\Delta\bar x_d/\Delta\bar\theta_{x,s}$. The differences are between the two exposures; for commands $+u$ and $-u$, the applied difference is $2u$.

In the actual microscope, the lenses can rotate the beam, so $A$ and $B$ are $2\times2$ blocks,

$$
\begin{pmatrix}\Delta\bar x_d\\\Delta\bar y_d\end{pmatrix}
=
\underbrace{\begin{pmatrix}A_{xx}&A_{xy}\\A_{yx}&A_{yy}\end{pmatrix}}_{A}
\begin{pmatrix}\Delta\bar x_s\\\Delta\bar y_s\end{pmatrix}
+
\underbrace{\begin{pmatrix}B_{xx}&B_{xy}\\B_{yx}&B_{yy}\end{pmatrix}}_{B}
\begin{pmatrix}\Delta\bar\theta_{x,s}\\\Delta\bar\theta_{y,s}\end{pmatrix}.
$$

An X shift measures the first column of $A$; X and Y tilts measure the two columns of $B$. The selected acquisition uses those three kinds of steering, with the round-lens model relating the remaining entries.

### 2. Variances and covariance measure the beam ellipse

For background-subtracted counts $w_p$ at detector pixel $p$, the measured quantities are

$$
\begin{aligned}
N&=\sum_p w_p,\qquad
\bar x_d=\frac{\sum_p w_px_p}{N},\qquad
\bar y_d=\frac{\sum_p w_py_p}{N},\\[4pt]
\operatorname{var}(x_d)&=\frac{\sum_p w_p(x_p-\bar x_d)^2}{N},\\
\operatorname{var}(y_d)&=\frac{\sum_p w_p(y_p-\bar y_d)^2}{N},\\
\operatorname{cov}(x_d,y_d)&=\frac{\sum_p w_p(x_p-\bar x_d)(y_p-\bar y_d)}{N}.
\end{aligned}
$$

The variances describe squared widths, while covariance describes how horizontal and vertical displacements vary together. They form the measured spatial covariance matrix,

$$
\Sigma_d=
\begin{pmatrix}
\operatorname{var}(x_d)&\operatorname{cov}(x_d,y_d)\\
\operatorname{cov}(x_d,y_d)&\operatorname{var}(y_d)
\end{pmatrix}.
$$

Its eigenvectors give the ellipse's directions, and the square roots of its eigenvalues give the RMS widths along those directions. For an aperture-clipped beam, this ellipse summarizes its second moments.

**Angles influence these measured widths through propagation.** In one transverse direction,

$$
\boxed{
\operatorname{var}(x_d)
=A^2\operatorname{var}(x_s)
+2AB\operatorname{cov}(x_s,\theta_{x,s})
+B^2\operatorname{var}(\theta_{x,s})
+\sigma_{\mathrm{det},x}^2.
}
$$

The $A^2$ term transports the specimen width, the $B^2$ term converts angular spread into detector width, and the $AB$ term accounts for whether the specimen beam is converging or diverging. Position–angle covariance is part of the model's beam description; the camera measures only positions.

For the actual two-direction calculation, define the specimen ray coordinates and the two rows that predict detector position,

$$
\mathbf q_s=(x_s,y_s,\theta_{x,s},\theta_{y,s})^\mathsf T,
\qquad \Sigma_s=\operatorname{Cov}(\mathbf q_s),
$$

$$
H=(A\;B)=\begin{pmatrix}\mathbf h_x\\\mathbf h_y\end{pmatrix},\qquad
\mathbf h_x=(A_{xx},A_{xy},B_{xx},B_{xy}),\quad
\mathbf h_y=(A_{yx},A_{yy},B_{yx},B_{yy}).
$$

The three shape predictions fitted to each image are exactly the entries of

$$
\boxed{\Sigma_d=H\Sigma_sH^\mathsf T+\Sigma_{\mathrm{det}}.}
$$

Equivalently,

$$
\begin{aligned}
\operatorname{var}(x_d)&=\mathbf h_x\Sigma_s\mathbf h_x^\mathsf T+(\Sigma_{\mathrm{det}})_{xx},\\
\operatorname{var}(y_d)&=\mathbf h_y\Sigma_s\mathbf h_y^\mathsf T+(\Sigma_{\mathrm{det}})_{yy},\\
\operatorname{cov}(x_d,y_d)&=\mathbf h_x\Sigma_s\mathbf h_y^\mathsf T+(\Sigma_{\mathrm{det}})_{xy}.
\end{aligned}
$$

Here $\Sigma_s$ is predicted from the source, condenser, and inserted aperture. It includes position–angle and angle–angle covariances that are inferred through their effect on the images. The equations above use physical detector distances; the implementation converts them to pixels and adds the known blur and pixel-sampling correction. The current isotropic detector model adds equal diagonal variances and zero XY covariance.

### 3. Intensity measures how much beam passes through the aperture

A candidate microscope predicts the beam density $\rho_a(x,y)$ at the aperture. Its transmitted fraction is

$$
T_a=\int\rho_a(x,y)\,|t_a(x,y)|^2\,dx\,dy,
\qquad
\boxed{N_a^{\mathrm{pred}}=N_{\mathrm{in}}T_a,}
$$

where $\rho_a$ is normalized to one, $t_a$ is the aperture's amplitude transmission, and $N_{\mathrm{in}}$ is the unknown incident electron count. A beam that overfills the aperture has a lower transmitted fraction.

The connection to $A$ and $B$ is visible in a simple example. For a centred, round Gaussian source with uncorrelated positions and angles, the beam width at the aperture is

$$
\sigma_a^2=A_a^2\sigma_0^2+B_a^2\sigma_{\theta,0}^2.
$$

Here $A_a,B_a$ describe **source-to-aperture** transport; the $A,B$ in the detector-position equations describe **specimen-to-detector** transport. Both maps are built from the fitted lens strengths and distances. An ideal circular aperture of radius $r$ then transmits

$$
\boxed{
T(r)=1-\exp\!\left[-\frac{r^2}{2\sigma_a^2}\right]
=1-\exp\!\left[-\frac{r^2}{2(A_a^2\sigma_0^2+B_a^2\sigma_{\theta,0}^2)}\right].
}
$$

The intensity therefore constrains a combination of upstream matrix entries and source spreads. The actual fit computes the integral numerically, including the smooth aperture edge and offsets, and also propagates the aperture's effect on the beam shape.

For the 20 and 50 µm aperture images at the same setting,

$$
N_{20}^{\mathrm{pred}}=N_{\mathrm{in}}T_{20},\qquad
N_{50}^{\mathrm{pred}}=N_{\mathrm{in}}T_{50},\qquad
\frac{N_{20}^{\mathrm{pred}}}{N_{50}^{\mathrm{pred}}}=\frac{T_{20}}{T_{50}}.
$$

The fit chooses the best common $N_{\mathrm{in}}$ analytically at every iteration, then compares the predicted counts with both measurements. For example, $T_{20}=0.4$ and $T_{50}=0.9$ can match counts of 400 and 900 with $N_{\mathrm{in}}=1000$, but no common incident count makes that candidate match 400 and 600. The microscope parameters must change to predict a different relative transmission.

This comparison requires equal incident dose between the two aperture exposures, or a known correction. The implementation fits the counts with their full measurement covariance; the ratio above explains why an absolute incident-intensity calibration is unnecessary. A single-aperture projector exposure has its own unknown incident dose, so its total count supplies no separate transmission constraint.

### How the measurements work together

| Measured from the images | What the model must reproduce |
|---|---|
| Beam-centre changes under known shifts and tilts | The specimen-to-detector $A,B$ response |
| Horizontal and vertical variances and XY covariance | The transported beam ellipse, including the effect of angular spread |
| Relative counts through the two apertures | How the upstream beam fills and aligns with the aperture |

All three predictions use the same fitted microscope and source parameters across the recorded settings. The fit varies those parameters, reconstructs the transfer matrices, and compares the six predicted quantities with the six measured quantities using their uncertainties and correlations. Individual images constrain combinations of parameters; changing the lens currents, voltage, aperture, and steering helps separate those combinations.